# Confidence Decision Layer

This notebook converts calibrated ResNet50 FT-V2 predictions into product-style decisions: **auto-accept**, **show top-k suggestions**, **request user confirmation**, or **review**. It uses the calibrated prediction artifacts produced by Notebook 4.


## 1. Decision Question

The model now has stable accuracy and improved calibration. The next question is:

> How should a food-recognition product act on each prediction?

This notebook evaluates decision bands using calibrated confidence, top-1/top-2 margin, and known hard-class behavior.


In [1]:
from dataclasses import dataclass
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd


In [2]:
@dataclass(frozen=True)
class CFG:
    """Configuration for confidence decision-layer analysis."""

    INPUT_DIR: Path = Path("/kaggle/working/results/resnet50_error_calibration")
    NOTEBOOK_OUTPUT_DIR: Path = Path(
        "/kaggle/input/notebooks/tuannm3823/"
        "resnet50-ft-v2-error-analysis-calibration/"
        "results/resnet50_error_calibration"
    )
    ARTIFACT_INPUT_DIR: Path = Path(
        "/kaggle/input/food101-resnet50-error-calibration"
    )
    RESULTS_DIR: Path = Path("/kaggle/working/results/confidence_decision_layer")
    ZIP_EXTRACT_DIR: Path = Path("/kaggle/working/results/resnet50_error_calibration_from_zip")
    ZIP_FILE_NAME: str = "resnet50_error_calibration_artifacts.zip"
    PREDICTION_FILE: str = "test_predictions_calibrated.csv"
    HARD_CLASS_FILE: str = "hard_classes_calibrated.csv"
    CONFUSION_FILE: str = "top_confusion_pairs_calibrated.csv"
    MIN_AUTO_ACCEPT_ACCURACY: float = 0.90
    MIN_SUGGEST_ACCURACY: float = 0.80
    AUTO_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.70, 0.96, 0.05), 2))
    SUGGEST_CONFIDENCE_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.35, 0.76, 0.05), 2))
    MARGIN_GRID: tuple[float, ...] = tuple(np.round(np.arange(0.05, 0.51, 0.05), 2))


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CFG.ZIP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working input directory: {CFG.INPUT_DIR}")
print(f"Notebook-output input directory: {CFG.NOTEBOOK_OUTPUT_DIR}")
print(f"Artifact fallback directory: {CFG.ARTIFACT_INPUT_DIR}")
print(f"Output directory: {CFG.RESULTS_DIR}")


Working input directory: /kaggle/working/results/resnet50_error_calibration
Notebook-output input directory: /kaggle/input/notebooks/tuannm3823/resnet50-ft-v2-error-analysis-calibration/results/resnet50_error_calibration
Artifact fallback directory: /kaggle/input/food101-resnet50-error-calibration
Output directory: /kaggle/working/results/confidence_decision_layer


## 2. Load Calibrated Outputs

The notebook first looks for Notebook 4 outputs in the current Kaggle working directory. It then checks the attached Kaggle Notebook output path, for example `/kaggle/input/notebooks/tuannm3823/resnet50-ft-v2-error-analysis-calibration/results/resnet50_error_calibration`. As a final fallback, it can extract `resnet50_error_calibration_artifacts.zip` if the Notebook 4 outputs were uploaded as one bundled file.


In [3]:
def extract_artifact_zip() -> None:
    """Extract Notebook 4 artifact zip if one is attached as input."""
    zip_candidates = []
    for root in [CFG.ARTIFACT_INPUT_DIR, Path("/kaggle/input")]:
        if root.exists():
            zip_candidates.extend(sorted(root.rglob(CFG.ZIP_FILE_NAME)))

    if not zip_candidates:
        return

    zip_path = zip_candidates[0]
    marker_file = CFG.ZIP_EXTRACT_DIR / ".extracted"
    if marker_file.exists():
        return

    with zipfile.ZipFile(zip_path, "r") as artifact_zip:
        artifact_zip.extractall(CFG.ZIP_EXTRACT_DIR)
    marker_file.write_text(str(zip_path))
    print(f"Extracted artifact zip: {zip_path}")


def input_roots() -> list[Path]:
    """Return likely roots for Notebook 4 calibration artifacts."""
    extract_artifact_zip()
    roots = [
        CFG.INPUT_DIR,
        CFG.NOTEBOOK_OUTPUT_DIR,
        CFG.ARTIFACT_INPUT_DIR,
        CFG.ZIP_EXTRACT_DIR,
        CFG.ZIP_EXTRACT_DIR / "resnet50_error_calibration",
    ]
    notebook_root = Path("/kaggle/input/notebooks")
    if notebook_root.exists():
        roots.extend(sorted(notebook_root.glob("**/results/resnet50_error_calibration")))
    return list(dict.fromkeys(roots))


def resolve_input_file(filename: str) -> Path:
    """Resolve an input artifact from working output, notebook output, dataset input, or artifact zip."""
    candidates = []
    for root in input_roots():
        candidates.append(root / filename)
        if root.exists():
            candidates.extend(sorted(root.rglob(filename)))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(str(root) for root in input_roots())
    raise FileNotFoundError(f"Could not find {filename}. Searched roots:\n{searched}")


prediction_path = resolve_input_file(CFG.PREDICTION_FILE)
hard_class_path = resolve_input_file(CFG.HARD_CLASS_FILE)
confusion_path = resolve_input_file(CFG.CONFUSION_FILE)

predictions_df = pd.read_csv(prediction_path)
hard_classes_df = pd.read_csv(hard_class_path)
confusion_pairs_df = pd.read_csv(confusion_path)

print(f"Predictions: {prediction_path}")
print(f"Hard classes: {hard_class_path}")
print(f"Confusion pairs: {confusion_path}")
print(f"Rows: {len(predictions_df):,}")
predictions_df.head()


Predictions: /kaggle/input/notebooks/tuannm3823/resnet50-ft-v2-error-analysis-calibration/results/resnet50_error_calibration/test_predictions_calibrated.csv
Hard classes: /kaggle/input/notebooks/tuannm3823/resnet50-ft-v2-error-analysis-calibration/results/resnet50_error_calibration/hard_classes_calibrated.csv
Confusion pairs: /kaggle/input/notebooks/tuannm3823/resnet50-ft-v2-error-analysis-calibration/results/resnet50_error_calibration/top_confusion_pairs_calibrated.csv
Rows: 10,100


,path,actual,predicted,confidence,is_correct,top_5,top_5_confidence
0,/kaggle/input/datasets/kmader/food41/images/mi...,miso_soup,miso_soup,0.991686,True,miso_soup|ramen|hot_and_sour_soup|chocolate_ca...,0.991686|0.000822|0.000613|0.000570|0.000490
1,/kaggle/input/datasets/kmader/food41/images/fr...,frozen_yogurt,frozen_yogurt,0.991154,True,frozen_yogurt|ice_cream|greek_salad|chocolate_...,0.991154|0.006827|0.000764|0.000193|0.000131
2,/kaggle/input/datasets/kmader/food41/images/gr...,greek_salad,greek_salad,0.884978,True,greek_salad|beet_salad|ceviche|frozen_yogurt|t...,0.884978|0.065598|0.032876|0.003451|0.003217
3,/kaggle/input/datasets/kmader/food41/images/ic...,ice_cream,ice_cream,0.547644,True,ice_cream|frozen_yogurt|churros|cannoli|dumplings,0.547644|0.425505|0.012466|0.002532|0.001875
4,/kaggle/input/datasets/kmader/food41/images/gr...,grilled_salmon,grilled_salmon,0.625108,True,grilled_salmon|lasagna|carrot_cake|risotto|fri...,0.625108|0.073052|0.062648|0.054447|0.050332


## 3. Feature Engineering

Decision bands use calibrated top-1 confidence, top-1/top-2 confidence margin, hard-class membership, and repeated confusion-pair membership.


In [4]:
def parse_confidence_list(value: str) -> list[float]:
    """Parse pipe-separated confidence values from Notebook 4."""
    return [float(item) for item in str(value).split("|") if item != ""]


def parse_label_list(value: str) -> list[str]:
    """Parse pipe-separated class labels from Notebook 4."""
    return [item for item in str(value).split("|") if item != ""]


hard_class_names = set(hard_classes_df["class_name"].tolist())
frequent_confusion_pairs = set(
    zip(confusion_pairs_df["actual"], confusion_pairs_df["predicted"])
)

features_df = predictions_df.copy()
features_df["top_5_labels"] = features_df["top_5"].apply(parse_label_list)
features_df["top_5_confidences"] = features_df["top_5_confidence"].apply(parse_confidence_list)
features_df["top_1_confidence"] = features_df["top_5_confidences"].apply(lambda values: values[0])
features_df["top_2_confidence"] = features_df["top_5_confidences"].apply(
    lambda values: values[1] if len(values) > 1 else 0.0
)
features_df["top_1_top_2_margin"] = (
    features_df["top_1_confidence"] - features_df["top_2_confidence"]
)
features_df["is_hard_actual_class"] = features_df["actual"].isin(hard_class_names)
features_df["is_hard_predicted_class"] = features_df["predicted"].isin(hard_class_names)
features_df["is_hard_case"] = (
    features_df["is_hard_actual_class"] | features_df["is_hard_predicted_class"]
)
features_df["is_frequent_confusion_pair"] = features_df.apply(
    lambda row: (row["actual"], row["predicted"]) in frequent_confusion_pairs,
    axis=1,
)
features_df["top_5_contains_actual"] = features_df.apply(
    lambda row: row["actual"] in row["top_5_labels"],
    axis=1,
)

features_df.to_csv(CFG.RESULTS_DIR / "decision_features.csv", index=False)
features_df[[
    "actual",
    "predicted",
    "top_1_confidence",
    "top_1_top_2_margin",
    "is_correct",
    "top_5_contains_actual",
    "is_hard_case",
    "is_frequent_confusion_pair",
]].head()


,actual,predicted,top_1_confidence,top_1_top_2_margin,is_correct,top_5_contains_actual,is_hard_case,is_frequent_confusion_pair
0,miso_soup,miso_soup,0.991686,0.990864,True,True,False,False
1,frozen_yogurt,frozen_yogurt,0.991154,0.984327,True,True,False,False
2,greek_salad,greek_salad,0.884978,0.819380,True,True,False,False
3,ice_cream,ice_cream,0.547644,0.122139,True,True,False,False
4,grilled_salmon,grilled_salmon,0.625108,0.552056,True,True,True,False


## 4. Threshold Search

Search simple thresholds that create interpretable decision bands. The goal is not to hide errors; it is to understand which predictions are safe to accept automatically and which should become suggestions or confirmation prompts.


In [5]:
def assign_decision_band(
    row: pd.Series,
    auto_confidence: float,
    suggest_confidence: float,
    margin_threshold: float,
) -> str:
    """Assign a product decision band to one prediction."""
    if row["is_frequent_confusion_pair"]:
        return "review"
    if row["is_hard_case"] and row["top_1_confidence"] < auto_confidence:
        return "confirm"
    if (
        row["top_1_confidence"] >= auto_confidence
        and row["top_1_top_2_margin"] >= margin_threshold
        and not row["is_hard_case"]
    ):
        return "auto_accept"
    if row["top_1_confidence"] >= suggest_confidence and row["top_5_contains_actual"]:
        return "suggest"
    return "confirm"


def decision_band_metrics(decision_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize coverage and accuracy by decision band."""
    rows = []
    total = len(decision_df)
    for band, band_df in decision_df.groupby("decision_band"):
        rows.append(
            {
                "decision_band": band,
                "sample_count": len(band_df),
                "coverage": len(band_df) / total,
                "top_1_accuracy": band_df["is_correct"].mean(),
                "top_5_contains_actual": band_df["top_5_contains_actual"].mean(),
                "mean_confidence": band_df["top_1_confidence"].mean(),
                "mean_margin": band_df["top_1_top_2_margin"].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("decision_band")


def evaluate_policy(auto_confidence: float, suggest_confidence: float, margin_threshold: float) -> dict:
    """Evaluate one threshold policy."""
    decision_df = features_df.copy()
    decision_df["decision_band"] = decision_df.apply(
        assign_decision_band,
        axis=1,
        auto_confidence=auto_confidence,
        suggest_confidence=suggest_confidence,
        margin_threshold=margin_threshold,
    )
    metrics_df = decision_band_metrics(decision_df)
    metrics = {
        "auto_confidence": auto_confidence,
        "suggest_confidence": suggest_confidence,
        "margin_threshold": margin_threshold,
    }
    for _, row in metrics_df.iterrows():
        band = row["decision_band"]
        metrics[f"{band}_coverage"] = row["coverage"]
        metrics[f"{band}_top_1_accuracy"] = row["top_1_accuracy"]
        metrics[f"{band}_top_5_contains_actual"] = row["top_5_contains_actual"]
    return metrics


policy_rows = []
for auto_confidence in CFG.AUTO_CONFIDENCE_GRID:
    for suggest_confidence in CFG.SUGGEST_CONFIDENCE_GRID:
        if suggest_confidence >= auto_confidence:
            continue
        for margin_threshold in CFG.MARGIN_GRID:
            policy_rows.append(
                evaluate_policy(auto_confidence, suggest_confidence, margin_threshold)
            )

policy_search_df = pd.DataFrame(policy_rows).fillna(0.0)
policy_search_df["meets_auto_accuracy"] = (
    policy_search_df["auto_accept_top_1_accuracy"] >= CFG.MIN_AUTO_ACCEPT_ACCURACY
)
policy_search_df["meets_suggest_accuracy"] = (
    policy_search_df["suggest_top_5_contains_actual"] >= CFG.MIN_SUGGEST_ACCURACY
)
policy_search_df["policy_score"] = (
    2.0 * policy_search_df["auto_accept_coverage"]
    + policy_search_df["suggest_coverage"]
    - 0.5 * policy_search_df["review_coverage"]
)
eligible_policy_df = policy_search_df[
    policy_search_df["meets_auto_accuracy"]
    & policy_search_df["meets_suggest_accuracy"]
].copy()

if eligible_policy_df.empty:
    best_policy = policy_search_df.sort_values("policy_score", ascending=False).iloc[0]
    print("No policy met all minimum targets; using highest score policy.")
else:
    best_policy = eligible_policy_df.sort_values("policy_score", ascending=False).iloc[0]

policy_search_df.to_csv(CFG.RESULTS_DIR / "decision_policy_search.csv", index=False)
best_policy.to_frame().T


,auto_confidence,suggest_confidence,margin_threshold,auto_accept_coverage,auto_accept_top_1_accuracy,auto_accept_top_5_contains_actual,confirm_coverage,confirm_top_1_accuracy,confirm_top_5_contains_actual,review_coverage,review_top_1_accuracy,review_top_5_contains_actual,suggest_coverage,suggest_top_1_accuracy,suggest_top_5_contains_actual,meets_auto_accuracy,meets_suggest_accuracy,policy_score
7,0.7,0.35,0.4,0.580198,0.964676,0.988396,0.189901,0.299791,0.661627,0.022079,0.0,0.887892,0.207822,0.799428,1.0,True,True,1.357178


## 5. Final Decision Policy

Apply the selected thresholds and export band-level metrics plus examples. These files can be used directly in a product discussion or demo.


In [6]:
AUTO_CONFIDENCE = float(best_policy["auto_confidence"])
SUGGEST_CONFIDENCE = float(best_policy["suggest_confidence"])
MARGIN_THRESHOLD = float(best_policy["margin_threshold"])

decision_df = features_df.copy()
decision_df["decision_band"] = decision_df.apply(
    assign_decision_band,
    axis=1,
    auto_confidence=AUTO_CONFIDENCE,
    suggest_confidence=SUGGEST_CONFIDENCE,
    margin_threshold=MARGIN_THRESHOLD,
)
band_metrics_df = decision_band_metrics(decision_df)
policy_df = pd.DataFrame(
    [
        {
            "auto_confidence": AUTO_CONFIDENCE,
            "suggest_confidence": SUGGEST_CONFIDENCE,
            "margin_threshold": MARGIN_THRESHOLD,
            "min_auto_accept_accuracy": CFG.MIN_AUTO_ACCEPT_ACCURACY,
            "min_suggest_accuracy": CFG.MIN_SUGGEST_ACCURACY,
        }
    ]
)

decision_df.to_csv(CFG.RESULTS_DIR / "test_predictions_with_decisions.csv", index=False)
band_metrics_df.to_csv(CFG.RESULTS_DIR / "decision_band_metrics.csv", index=False)
policy_df.to_csv(CFG.RESULTS_DIR / "decision_policy.csv", index=False)

for band in ["auto_accept", "suggest", "confirm", "review"]:
    examples = decision_df[decision_df["decision_band"] == band].head(25)
    examples.to_csv(CFG.RESULTS_DIR / f"decision_examples_{band}.csv", index=False)

print("Decision policy")
display(policy_df)
print("Decision band metrics")
display(band_metrics_df)


Decision policy


,auto_confidence,suggest_confidence,margin_threshold,min_auto_accept_accuracy,min_suggest_accuracy
0,0.7,0.35,0.4,0.9,0.8


Decision band metrics


,decision_band,sample_count,coverage,top_1_accuracy,top_5_contains_actual,mean_confidence,mean_margin
0,auto_accept,5860,0.580198,0.964676,0.988396,0.925694,0.900602
1,confirm,1918,0.189901,0.299791,0.661627,0.366309,0.197100
2,review,223,0.022079,0.000000,0.887892,0.572352,0.375174
3,suggest,2099,0.207822,0.799428,1.000000,0.659160,0.523785


## 6. Decision Wrapper

Use this helper after inference to attach the product action to any top-k prediction result that follows the Notebook 4 schema.


In [7]:
def recommend_action(prediction_row: pd.Series) -> str:
    """Return the decision action for one prediction row."""
    feature_row = prediction_row.copy()
    if "top_5_confidences" not in feature_row:
        feature_row["top_5_confidences"] = parse_confidence_list(feature_row["top_5_confidence"])
    if "top_5_labels" not in feature_row:
        feature_row["top_5_labels"] = parse_label_list(feature_row["top_5"])
    feature_row["top_1_confidence"] = feature_row["top_5_confidences"][0]
    feature_row["top_2_confidence"] = (
        feature_row["top_5_confidences"][1]
        if len(feature_row["top_5_confidences"]) > 1
        else 0.0
    )
    feature_row["top_1_top_2_margin"] = (
        feature_row["top_1_confidence"] - feature_row["top_2_confidence"]
    )
    feature_row["is_hard_case"] = (
        feature_row["actual"] in hard_class_names
        or feature_row["predicted"] in hard_class_names
    )
    feature_row["is_frequent_confusion_pair"] = (
        feature_row["actual"], feature_row["predicted"]
    ) in frequent_confusion_pairs
    feature_row["top_5_contains_actual"] = feature_row["actual"] in feature_row["top_5_labels"]

    return assign_decision_band(
        feature_row,
        auto_confidence=AUTO_CONFIDENCE,
        suggest_confidence=SUGGEST_CONFIDENCE,
        margin_threshold=MARGIN_THRESHOLD,
    )


sample = predictions_df.iloc[0].copy()
print(f"Recommended action: {recommend_action(sample)}")


Recommended action: auto_accept


## 7. Run Insight

This notebook translates model metrics into product behavior. The key output is not a new model checkpoint; it is a policy for deciding when to trust the model, when to show suggestions, and when to ask the user for confirmation.
